# Fine-tune EDM CIFAR-10 → CIFAR-100

Fine-tunes NVIDIA's pretrained CIFAR-10 EDM diffusion model on CIFAR-100, following the [MimicDiffusion](https://github.com/psky1111/MimicDiffusion) recipe: fine-tune the **unconditional** EDM checkpoint. This avoids the class-count mismatch you'd get fine-tuning the conditional checkpoint (10-way label embedding vs. CIFAR-100's 100 classes).

**Before running:** `Runtime` → `Change runtime type` → select a GPU (T4 or better).

**Persistence:** set `USE_DRIVE = True` in the config cell and mount Drive to keep the repo clone, prepared dataset zip, downloaded checkpoint, and training runs/snapshots on Drive so nothing is lost if the Colab session terminates -- no need to "Save a copy in Drive" of this notebook itself, since it's tracked on GitHub. The one thing deliberately kept off Drive is the raw 50k-PNG CIFAR-100 scratch dump: it's disposable (regenerated from torchvision in under a minute) and Drive's per-file overhead makes writing that many small files there very slow. If training gets interrupted mid-run, just re-run the fine-tuning cell -- it auto-detects the latest checkpoint under `outdir` on Drive and resumes from there instead of starting over.

Source of truth for this workflow: [`finetune_edm_cifar100.py`](./finetune_edm_cifar100.py) in this repo. If you hit an issue running this notebook, report it back in the Claude Code session that generated it -- fixes land in that `.py` file (and this notebook) and get pushed to this branch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import glob
import os
import torchvision

# ── Configuration ──
USE_DRIVE = True  # read/write everything under Google Drive
drive_base_path = '/content/drive/MyDrive/FineTunedCheckpoint/edm-cifar100'  # only used if USE_DRIVE
# Paths below are quoted before being passed to shell magics, so spaces are
# tolerated -- but note this path is unrelated to where the notebook file
# itself lives; it's just where this script writes its own working files.

BASE = drive_base_path if USE_DRIVE else '/content'
# NOTE: BASE is not created here -- see the next cell. Creating
# '/content/drive/...' locally before Drive is mounted there makes
# drive.mount() refuse to mount ("Mountpoint must not already contain
# files"), since it then finds a non-empty local directory sitting at the
# mount point instead of an empty one.

COND = False  # unconditional fine-tuning (recommended -- see markdown cell above).
# COND=True is possible but the label-embedding weights won't transfer
# (shape mismatch: 10 classes -> 100), so --transfer effectively
# reinitializes that layer at random; only the backbone benefits.

DURATION_MIMG = 10  # fine-tuning budget, in millions of images
                    # (the paper's from-scratch CIFAR-10 run used 200; fine-tuning needs far less)
BATCH = 128         # lower than the paper's default of 512 to fit a single Colab GPU;
                    # scale --tick/--snap in the training cell if you raise this a lot

In [ ]:
# Mount Drive (only if USE_DRIVE=True above), THEN create BASE.
# Order matters: creating BASE before this would create a local stub at
# /content/drive that blocks the mount (see note in the cell above).
if USE_DRIVE:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    else:
        print("Google Drive is already mounted at /content/drive")

os.makedirs(BASE, exist_ok=True)

In [ ]:
# 1. Clone the NVlabs/edm repository
edm_dir = os.path.join(BASE, 'edm')
if not os.path.isdir(edm_dir):
    !git clone https://github.com/NVlabs/edm.git "{edm_dir}"

%cd "{edm_dir}"

In [ ]:
# 2. Install dependencies.
# The repo ships environment.yml (a conda spec), not a pip requirements.txt
# -- `pip install -r environment.yml` fails trying to parse YAML as
# requirements. Colab already has a CUDA-matched torch/numpy/pillow/scipy,
# so we deliberately don't force environment.yml's pinned torch==1.12.1
# (that would fight the preinstalled CUDA build for no benefit); just
# pip-install the packages listed there.
!pip install "numpy>=1.20" "click>=8.0" "pillow>=8.3.1" "scipy>=1.7.1" psutil requests tqdm imageio "imageio-ffmpeg>=0.4.3" pyspng

In [ ]:
# 3. Prepare the CIFAR-100 dataset.
# dataset_tool.py only derives labels from a dataset.json or from top-level
# subfolder names -- NOT from filenames -- so save each image into a
# per-class subfolder (train/<label>/*.png) rather than one flat folder.
#
# The raw PNG dump is kept on local disk (not under BASE) even when
# USE_DRIVE=True: it's disposable scratch input to dataset_tool.py below,
# and writing ~50k individual files through Drive's FUSE mount is slow.
local_scratch = '/content/cifar100-scratch'
raw_dir = os.path.join(local_scratch, 'train')
os.makedirs(raw_dir, exist_ok=True)

dataset_zip = os.path.join(BASE, 'datasets', 'cifar100-32x32.zip')
os.makedirs(os.path.dirname(dataset_zip), exist_ok=True)

if os.path.isfile(dataset_zip):
    # Already prepared and persisted (e.g. from a previous session on Drive)
    print(f"Found existing prepared dataset at {dataset_zip}, skipping re-prep.")
else:
    trainset = torchvision.datasets.CIFAR100(
        root=os.path.join(local_scratch, 'data_temp'), train=True, download=True,
    )
    for i, (img, label) in enumerate(trainset):
        class_dir = os.path.join(raw_dir, f'{label:03d}')
        os.makedirs(class_dir, exist_ok=True)
        img.save(os.path.join(class_dir, f'{i:05d}.png'))
    print(f"CIFAR-100 training images saved to {raw_dir}")

    print(f"Converting to EDM dataset format ({dataset_zip})...")
    !python dataset_tool.py --source="{raw_dir}" --dest="{dataset_zip}" --resolution=32x32

In [ ]:
# 4. Download the pretrained EDM CIFAR-10 checkpoint (NVIDIA ships .pkl, not .pt)
checkpoints_dir = os.path.join(BASE, 'checkpoints')
os.makedirs(checkpoints_dir, exist_ok=True)
ckpt_name = 'edm-cifar10-32x32-cond-vp.pkl' if COND else 'edm-cifar10-32x32-uncond-vp.pkl'
ckpt_path = os.path.join(checkpoints_dir, ckpt_name)
!wget -nc https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/{ckpt_name} -P "{checkpoints_dir}"

In [ ]:
# 5. Fine-tune.
outdir = os.path.join(BASE, 'training-runs-cifar100')
os.makedirs(outdir, exist_ok=True)

# If a previous run under outdir got interrupted (Colab disconnect, runtime
# recycle, etc.), --dump periodically wrote a full training-state file
# (optimizer + step count, not just weights) there. Pick the latest one and
# --resume from it instead of --transfer-ing the pretrained checkpoint again
# -- that's what actually avoids losing progress on session termination.
resume_candidates = sorted(glob.glob(os.path.join(outdir, '*', 'training-state-*.pt')))
resume_path = resume_candidates[-1] if resume_candidates else None

if resume_path:
    print(f"Found existing training state, resuming from {resume_path}")
    weight_arg = f'--resume="{resume_path}"'
else:
    print(f"No existing training state found, transferring pretrained weights from {ckpt_path}")
    weight_arg = f'--transfer="{ckpt_path}"'

# --duration is in millions of images, not kimg. --tick/--snap are scaled
# down from the (50, 50) defaults so a short fine-tuning run still produces
# a handful of checkpoints, not just one at the very end.
train_cmd = (
    f'python train.py '
    f'--outdir="{outdir}" '
    f'--data="{dataset_zip}" '
    f'--cond={"1" if COND else "0"} '
    f'{weight_arg} '
    f'--duration={DURATION_MIMG} '
    f'--batch={BATCH} '
    f'--tick=10 '
    f'--snap=10 '
    f'--dump=10 '
    f'--metrics=none'
)
print(train_cmd)
!{train_cmd}

In [ ]:
print("Fine-tuning finished. Network snapshots (.pkl) are under:")
print(f"  {outdir}")